# Train the cartoon colorizer on Colab

Runtime → Change runtime type → **T4 GPU**, then run each cell in order.

In [ ]:
!git clone https://github.com/ArunaSamhitha/Cartoon-Colorization-Using-CycleGAN.git
%cd Cartoon-Colorization-Using-CycleGAN
!pip install -q -r requirements.txt
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 1. Get frames

Upload cartoon video clips (or a zip of color frames) to Google Drive, mount it, and point the paths below at them. A few thousand frames is plenty.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

VIDEOS = '/content/drive/MyDrive/cartoon_clips'   # folder of .mp4 files
!python scripts/extract_frames.py --videos "$VIDEOS" --out data/frames --every 12

## 2. Train

Checkpoints go to Drive so a disconnected session can resume with `--resume`. For a quick first run, try `--image-size 128 --n-blocks 6 --epochs 40 --decay-start 20`.

In [ ]:
OUT = '/content/drive/MyDrive/cyclegan_runs/exp1'
!python train.py --frames data/frames --out "$OUT" --epochs 100 --decay-start 50
# To continue after a disconnect:
# !python train.py --frames data/frames --out "$OUT" --epochs 100 --decay-start 50 --resume "$OUT/checkpoints/last.pt"

## 3. Look at progress

In [ ]:
import glob
from IPython.display import Image, display
last = sorted(glob.glob(f'{OUT}/samples/*.png'))[-1]
print(last); display(Image(last))

## 4. Colorize a clip and measure FPS

The FPS printed here is the number to put in the README.

In [ ]:
!python colorize.py --ckpt "$OUT/checkpoints/colorizer.pt" --input "/content/drive/MyDrive/bw_test_clip.mp4" --output /content/colorized.mp4

## 5. Save a results image for the README

Copy a good sample grid into `assets/results.png` in the repo and uncomment the image line at the top of the README.